# Three-dimensional conservative RAD: local P4 and face P1

The oscillatory problem from Araya et al. (2024), section 5.2.1, uses K=0.1 I, beta=(1,0,0), and u=sin(6 pi x) sin(4 pi y) sin(2 pi z). This study preserves those coefficients and polynomial degrees, with explicitly different deterministic tetrahedral meshes. The classical P2 comparison has its own exact-solution refinement check.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from pymhm.tetrahedral import TetraMesh
from pymhm.rad3d import solve_rad_3d
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent


## Small affine transport patch

The Robin multiplier is (-K grad(u)+beta u/2).n, whereas the physical conservative flux is -K grad(u)+beta u. Weak Dirichlet data prescribe face moments.


In [ ]:
def affine(x):
    """Return a manufactured affine scalar with nonhomogeneous boundary data."""
    return 1 + x @ np.array([1.0, 2.0, 3.0])
solution = solve_rad_3d(TetraMesh.unit_cube(1), degree=4, local_refinement=1, velocity=[1, 0, 0], source=1.0, dirichlet=affine)
assert solution.l2_error(affine) < 1e-10
assert solution.h1_seminorm_error([1, 2, 3]) < 1e-9
print("Affine P4 / P1-face patch passed")


## Five-level analytical campaign

Scalar, gradient, and full physical flux norms use volume integration. The V norm includes L2/diameter squared, with diameter squared equal to three. Assembly and error quadrature use order eight; the finest errors are checked independently with order ten.


In [ ]:
record = json.loads((root / "examples/results/rad3d.json").read_text())
assert not record["source_changed_during_run"]
assert [row["n"] for row in record["rows"]] == [1, 2, 3, 4, 5]
for method in ("mhm", "classical"):
    rows = record["rows"]
    assert rows[-1][method]["V_error"] < rows[0][method]["V_error"]
    assert max(row[method]["backward_residual"] for row in rows) < 1e-10
    for norm in ("l2_error", "V_error", "physical_flux_l2_error"):
        np.testing.assert_allclose(rows[-1][method][norm], rows[-1]["quadrature_order10"][method][norm], rtol=2e-6, atol=1e-12)
    print(method, [(r["n"], r[method]["l2_error"], r[method]["V_error"]) for r in rows])
last = record["rows"][-1]
archive = root / "examples/results" / last["fields"]
assert hashlib.sha256(archive.read_bytes()).hexdigest() == last["fields_sha256"]


In [ ]:
for name in ("convergence", "scalar", "flux"):
    display(Image(filename=str(root / f"docs/figures/rad3d/{name}.png")))


The field panels keep individual one-sided tetrahedral values and show the actual macro intersections on every map. The two methods use shared physical limits and centered difference limits. Native optional tests independently assemble P3/P4 Galerkin and SUPG matrices and loads in DOLFINx/UFL; source notebooks remain unexecuted.
